In [31]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [32]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf2を用いて分析を行う。
sc_x,df_yはdf2を基に作成する。
ただし、必要があればdf1も利用する。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')
df2 = pd.read_csv('datafiles/df2_after_drop.csv')

sc_x = df2.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df2['SalePrice'])

In [33]:
print(df2.columns)

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtUnfSF',
       ...
       'PoolQC_Fa', 'PoolQC_Gd', 'BsmtQual_Fa', 'BsmtQual_Gd', 'BsmtQual_NA',
       'BsmtQual_TA', 'LandContour_HLS', 'LandContour_Low', 'LandContour_Lvl',
       'SalePrice'],
      dtype='object', length=250)


In [34]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [35]:
#各種モデルの作成

#重回帰
model1 = LinearRegression()
model1.fit(sc_x, df_y)

#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

#ラッソ回帰
model3 = Lasso(alpha = 100)
model3.fit(sc_x, df_y)

#回帰木
model4 = DecisionTreeRegressor(max_depth = 10, random_state = 0)
model4.fit(sc_x, df_y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [36]:
#依然として多重共線性は高いため、今後も多重共線性の解消を目指す。

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)


,VIF_Factor,features
22,2178.616803,GarageYrBlt
107,115.837730,Exterior1st_VinylSd
113,283.746702,GarageCond_TA
123,2202.752816,GarageFinish_NA
182,105.805972,Exterior2nd_VinylSd
185,165.010225,RoofStyle_Gable
187,153.477931,RoofStyle_Hip
193,233.699047,GarageQual_TA
236,960.199106,MiscFeature_NA
238,802.650747,MiscFeature_Shed


In [37]:
#一列dropしてみる
sc_x = sc_x.drop(['GarageFinish_NA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
107,115.837134,Exterior1st_VinylSd
113,282.411676,GarageCond_TA
181,105.785392,Exterior2nd_VinylSd
184,164.851995,RoofStyle_Gable
186,153.344617,RoofStyle_Hip
192,233.384665,GarageQual_TA
235,959.675823,MiscFeature_NA
237,802.110365,MiscFeature_Shed


完成したmodel2のスコア＝0.7788228561257451


In [38]:
#一列dropしてみる
sc_x = sc_x.drop(['MiscFeature_NA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
107,115.678525,Exterior1st_VinylSd
113,282.329291,GarageCond_TA
181,105.733960,Exterior2nd_VinylSd
184,164.851570,RoofStyle_Gable
186,153.344550,RoofStyle_Hip
192,233.363856,GarageQual_TA


完成したmodel2のスコア＝0.7788175817571735


In [39]:
#3列dropしてみる
sc_x = sc_x.drop(['Exterior2nd_VinylSd', 'GarageCond_TA', 'RoofStyle_Gable'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features


完成したmodel2のスコア＝0.7785744518848133


In [40]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]

df_high_vif.sort_values('VIF_Factor', ascending=False)

,VIF_Factor,features
76,90.503234,ExterCond_TA
34,81.381629,GarageType_Attchd
74,76.363708,ExterCond_Gd
89,74.401726,RoofMatl_CompShg
38,65.809905,GarageType_Detchd
189,54.715176,GarageQual_TA
22,53.575630,GarageYrBlt
209,51.800551,SaleType_New
103,51.486680,Exterior1st_MetalSd
225,49.162602,MSZoning_RL


In [41]:
#3列dropしてみる
sc_x = sc_x.drop(['ExterCond_TA', 'GarageType_Attchd'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
df_high_vif.sort_values('VIF_Factor', ascending=False)

,VIF_Factor,features
87,74.294750,RoofMatl_CompShg
187,54.692519,GarageQual_TA
207,51.798480,SaleType_New
101,51.119943,Exterior1st_MetalSd
42,49.071613,SaleCondition_Partial
223,49.044799,MSZoning_RL
10,43.948652,TotalBsmtSF
128,40.008059,Heating_GasA
173,39.335555,Exterior2nd_MetalSd
8,37.071897,BsmtFinSF1


In [42]:
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7786656098208713


In [43]:
#4列dropしてみる
sc_x = sc_x.drop(['RoofMatl_CompShg', 'GarageQual_TA', 'SaleType_New', 'Exterior1st_MetalSd'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

,VIF_Factor,features
219,48.946791,MSZoning_RL
10,43.205565,TotalBsmtSF
126,39.953988,Heating_GasA
8,36.797813,BsmtFinSF1
9,35.658095,BsmtUnfSF
0,34.745525,MSSubClass
220,32.669914,MSZoning_RM
151,32.515975,MasVnrType_NA
97,28.835127,Exterior1st_CemntBd
150,28.079813,MasVnrType_BrkFace


In [44]:
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7786075560896846


In [45]:
#4列dropしてみる
sc_x = sc_x.drop(['MSZoning_RL', 'TotalBsmtSF', 'Heating_GasA', 'BsmtFinSF1'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

,VIF_Factor,features
0,34.608125,MSSubClass
148,32.438191,MasVnrType_NA
95,28.758254,Exterior1st_CemntBd
147,28.017892,MasVnrType_BrkFace
165,27.951783,Exterior2nd_CmentBd
112,27.019800,FireplaceQu_NA
11,24.884128,GrLivArea
55,22.784311,Neighborhood_NAmes
101,21.672533,Exterior1st_VinylSd
43,18.709625,ExterQual_TA


In [46]:
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7960218940921331


In [47]:
#vifが大きいものを選択
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 28]

#4列dropしてみる
to_drop = set(df_high_vif['features'])
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
110,27.008619,FireplaceQu_NA
10,24.705651,GrLivArea
54,22.475783,Neighborhood_NAmes
99,20.689968,Exterior1st_VinylSd
42,18.680007,ExterQual_TA
59,18.112330,Neighborhood_OldTown
94,15.836130,Exterior1st_HdBoard
162,15.789927,Exterior2nd_HdBoard
8,15.459261,1stFlrSF
109,15.360314,FireplaceQu_Gd


完成したmodel2のスコア＝0.795079666769221


In [48]:
#vifが大きいものを選択
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 18.5]

#4列dropしてみる
to_drop = set(df_high_vif['features'])
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
4,13.906200,YearBuilt
91,13.359459,Exterior1st_HdBoard
157,13.227892,Exterior2nd_HdBoard
177,11.707513,Condition2_Norm
221,11.399673,BsmtQual_TA
139,11.309360,BsmtFinType2_Unf
164,11.126744,Exterior2nd_Wd Sdng
96,10.833053,Exterior1st_Wd Sdng


完成したmodel2のスコア＝0.7971109145093969


In [49]:
#dropしてみる
sc_x = sc_x.drop(['YearBuilt', 'Exterior1st_HdBoard', 'Condition2_Norm', 'BsmtQual_TA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

KeyboardInterrupt: 

In [ ]:
#dropしてみる
sc_x = sc_x.drop(['BsmtFinType2_Unf'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 5]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
114,9.790953,Functional_Typ
121,9.712928,KitchenQual_TA
161,8.190216,Exterior2nd_Wd Sdng
18,7.643792,GarageCars
94,7.642908,Exterior1st_Wd Sdng
76,7.536515,Foundation_PConc
120,7.334261,KitchenQual_Gd
19,7.019464,GarageArea
7,6.936213,1stFlrSF
191,6.676330,HouseStyle_1Story


完成したmodel2のスコア＝0.7965658597529965


In [ ]:
#VIFの最大値が10以下になるよう特徴量を削除した場合、model2のスコアは0.796となった。改変前のスコアは0.777であった。


In [ ]:
#主成分分析にて多重共線性の解消を目指す
df1 = pd.read_csv('datafiles/df2_after_drop.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df1['SalePrice'])

In [ ]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

,VIF_Factor,features
123,2202.752816,GarageFinish_NA
22,2178.616803,GarageYrBlt
236,960.199106,MiscFeature_NA
238,802.650747,MiscFeature_Shed
113,283.746702,GarageCond_TA
...,...,...
53,11.641993,Neighborhood_Edwards
184,11.246911,Exterior2nd_Wd Shng
139,10.947886,KitchenQual_TA
134,10.738728,Heating_Grav


In [ ]:
sc_x.columns

Index([     'MSSubClass',     'LotFrontage',         'LotArea',
           'OverallQual',     'OverallCond',       'YearBuilt',
          'YearRemodAdd',      'MasVnrArea',      'BsmtFinSF1',
             'BsmtUnfSF',
       ...
             'PoolQC_Fa',       'PoolQC_Gd',     'BsmtQual_Fa',
           'BsmtQual_Gd',     'BsmtQual_NA',     'BsmtQual_TA',
       'LandContour_HLS', 'LandContour_Low', 'LandContour_Lvl',
                       0],
      dtype='object', length=247)

In [ ]:
#GarageFinish列をダミー変数化した列一覧をリスト化し、主成分分析により一列化
GarageFinish_cols = []
for c in sc_x.columns:
    if 'GarageFinish_' in c:
        GarageFinish_cols.append(c)
print(GarageFinish_cols)


TypeError: argument of type 'int' is not iterable

In [ ]:

PCAmodel = PCA(whiten = True)
GarageFinish_df = pd.DataFrame()
for c in GarageFinish_cols:
    GarageFinish_df = pd.concat([GarageFinish_df, sc_x[c]], axis = 1)
PCAmodel.fit(GarageFinish_df)
GarageFinish = PCAmodel.transform(sc_x[GarageFinish_cols])

#累積寄与率の閾値を0.8として、n_componentsを設定
thred = 0.8
final_num = 0
ratio =PCAmodel.explained_variance_ratio_
array = []
for i in range(len(ratio)):
    ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
    if ruiseki > thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
          final_num = i
          break
PCAmodel = PCA(n_components = final_num, whiten = True)
PCAmodel.fit(sc_x[GarageFinish_cols])
GarageFinish = PCAmodel.transform(sc_x[GarageFinish_cols])
GarageFinish = pd.DataFrame(GarageFinish)

for c in sc_x[GarageFinish_cols]:
    sc_x = sc_x.drop([c], axis = 1)
sc_x = pd.concat([sc_x, GarageFinish], axis = 1)





result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


In [ ]:
#主成分分析で列を減らす（多重共線性の解消も目指す）